# Scaling laws: Herding

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import json
import sys
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
import torch
from transformers import set_seed

cwd = Path.cwd().resolve()
if (cwd / "notebooks" / "dilm_wrapper.py").exists():
    ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks"
elif (cwd / "dilm_wrapper.py").exists():
    NOTEBOOK_DIR = cwd
    ROOT = cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {cwd}")

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(ROOT / "src"))

from dataset_attrs import DATASET_ATTRS
from learner import MODEL_ATTRS
from dilm_wrapper import (
    RESULTS_ROOT,
    build_coreset_module,
    build_data_module,
    build_evaluator,
    build_generator,
    build_learner,
    save_summary,
    summarize,
)

sns.set_theme(style="whitegrid")

In [ ]:
METHOD = "herding"
SEED = 42

TASKS = list(DATASET_ATTRS.keys())
LEARNER_MODELS = list(MODEL_ATTRS.keys())

DPC_GRID = [1, 5, 10, 20, 50, 100, 500, 1000]
N_DATASET = 1
SKIP_EXISTING = True

EVAL_KW = dict(
    train_step=50,
    batch_size=32,
    lr=1e-4,
    n_eval_per_dataset=1,
    bf16=False,
)

SCALING_ROOT = RESULTS_ROOT / "scaling_laws" / METHOD
SCALING_ROOT.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
mlflow.set_tracking_uri(f"file:{RESULTS_ROOT}/mlruns")
mlflow.set_experiment("scaling_laws.herding")

print("tasks:", TASKS)
print("models:", LEARNER_MODELS)
print("dpc:", DPC_GRID)

In [ ]:
def safe_name(value: str) -> str:
    return value.replace("/", "__")


def run_dir(task: str, learner_model: str, dpc: int) -> Path:
    return SCALING_ROOT / task / safe_name(learner_model) / f"dpc{dpc}_seed{SEED}"


def summary_path(task: str, learner_model: str, dpc: int) -> Path:
    return run_dir(task, learner_model, dpc) / "summary.json"


def load_summary(path: Path) -> dict:
    return json.loads(path.read_text())


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def run_task_model(task: str, learner_model: str) -> list[dict]:
    print(f"\n=== {task} | {learner_model} ===")
    rows = []

    learner = build_learner(learner_model, task, gradient_checkpointing=True)
    generator = build_generator(task, gradient_checkpointing=True)
    data_module = build_data_module(task, learner, generator, train_batch_size=64)
    evaluator = build_evaluator(task, **EVAL_KW)
    selector = build_coreset_module(task, METHOD, data_module)
    metric_key = evaluator.metric_key

    for dpc in DPC_GRID:
        path = summary_path(task, learner_model, dpc)
        if SKIP_EXISTING and path.exists():
            print(f"skip existing: {path}")
            rows.append(load_summary(path))
            continue

        k = dpc * data_module.num_labels
        rd = run_dir(task, learner_model, dpc)
        rd.mkdir(parents=True, exist_ok=True)
        print(f"run dpc={dpc}, k={k}")

        t0 = time.perf_counter()
        distilled = selector.generate_dataset(dpc=dpc, n=N_DATASET)
        selection_time_sec = time.perf_counter() - t0

        t1 = time.perf_counter()
        with mlflow.start_run(run_name=f"{METHOD}.{task}.{safe_name(learner_model)}.dpc{dpc}.seed{SEED}"):
            mlflow.log_params({
                "method": METHOD,
                "task": task,
                "learner": learner_model,
                "dpc": dpc,
                "k": k,
                "n_dataset": N_DATASET,
                "seed": SEED,
                **EVAL_KW,
            })
            results = evaluator.evaluate(
                dataset_list=distilled,
                learner=learner,
                data_module=data_module,
                save_result_dir=str(rd / "metrics"),
                verbose=True,
            )
        eval_time_sec = time.perf_counter() - t1

        summary = summarize(
            results,
            metric_key,
            name=f"{METHOD}_dpc{dpc}_seed{SEED}",
            method=METHOD,
            task=task,
            learner=learner_model,
            dpc=dpc,
            k=k,
            n_dataset=N_DATASET,
            seed=SEED,
            selection_time_sec=selection_time_sec,
            eval_time_sec=eval_time_sec,
        )
        save_summary(summary, rd)
        rows.append(summary)
        print({"metric": metric_key, "score": summary[f"{metric_key}_mean"], "eval_time_sec": round(eval_time_sec, 1)})

    del learner, generator, data_module, evaluator, selector
    cleanup_cuda()
    return rows

## Run

In [ ]:
all_rows = []
errors = []

for task in TASKS:
    for learner_model in LEARNER_MODELS:
        try:
            all_rows.extend(run_task_model(task, learner_model))
        except Exception as exc:
            cleanup_cuda()
            err = {
                "task": task,
                "learner": learner_model,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
            errors.append(err)
            err_dir = SCALING_ROOT / task / safe_name(learner_model)
            err_dir.mkdir(parents=True, exist_ok=True)
            (err_dir / "error.json").write_text(json.dumps(err, indent=2))
            print("FAILED", task, learner_model, type(exc).__name__, exc)

df = pd.DataFrame(all_rows)
csv_path = SCALING_ROOT / "scaling_laws_herding.csv"
df.to_csv(csv_path, index=False)
print("saved", csv_path)
print("errors", len(errors))
df

## Collect saved summaries

In [ ]:
rows = [json.loads(path.read_text()) for path in SCALING_ROOT.rglob("summary.json")]
df = pd.DataFrame(rows)
if not df.empty:
    df["score"] = df.apply(lambda row: row[f"{row['metric']}_mean"], axis=1)
    df = df.sort_values(["task", "learner", "dpc"])
    df.to_csv(SCALING_ROOT / "scaling_laws_herding.csv", index=False)
df

## Plot

In [ ]:
if df.empty:
    raise RuntimeError("No summaries found. Run previous cells first.")

plot_df = df.copy()
plot_df["learner"] = plot_df["learner"].astype(str)

g = sns.relplot(
    data=plot_df,
    x="k",
    y="score",
    hue="learner",
    col="task",
    kind="line",
    marker="o",
    facet_kws={"sharey": False, "sharex": False},
    height=4,
    aspect=1.1,
)
g.set_axis_labels("K = DPC * num_labels", "score")
g.set_titles("{col_name}")
g.fig.suptitle("Scaling laws for Herding", y=1.05)

plot_path = SCALING_ROOT / "scaling_laws_herding.png"
g.fig.savefig(plot_path, dpi=160, bbox_inches="tight")
print("saved", plot_path)